In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from config import get_engine

In [9]:
OUTPUT_DIR = Path('output')
DATA_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
engine = get_engine()

In [18]:
def load_data(engine):
    print("Loading 7 datasets...")

    files = {
        "customer_fact": "fact_customers.csv",
        "fact_usage_monthly": "fact_usage_monthly.csv",
        "fact_transaction": "fact_transactions.csv",
        "fact_engagement_events": "fact_engagement_events.csv",
        "dim_geography": "dim_geography.csv",
        "dim_industry": "dim_industry.csv",
        "dim_product": "dim_product.csv"
    }

    for table_name, file_name in files.items():
        print(f"Loading {file_name} -> {table_name}")

        df = pd.read_csv(DATA_DIR / file_name)

        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",  # use "append" if you don't want to overwrite
            index=False
        )

        print(f"✓ Loaded {len(df)} rows into {table_name}")

    print("All datasets loaded successfully!")

In [19]:
load_data(engine)

Loading 7 datasets...
Loading fact_customers.csv -> customer_fact
✓ Loaded 5150 rows into customer_fact
Loading fact_usage_monthly.csv -> fact_usage_monthly
✓ Loaded 131268 rows into fact_usage_monthly
Loading fact_transactions.csv -> fact_transaction
✓ Loaded 54858 rows into fact_transaction
Loading fact_engagement_events.csv -> fact_engagement_events
✓ Loaded 54822 rows into fact_engagement_events
Loading dim_geography.csv -> dim_geography
✓ Loaded 183 rows into dim_geography
Loading dim_industry.csv -> dim_industry
✓ Loaded 22 rows into dim_industry
Loading dim_product.csv -> dim_product
✓ Loaded 32 rows into dim_product
All datasets loaded successfully!


In [41]:

def load_from_postgres():
    print("Retrieving 7 tables from PostgreSQL...")

    # Your exact code here
    cust = pd.read_sql('SELECT * FROM customer_fact', engine)
    usage = pd.read_sql('SELECT * FROM fact_usage_monthly', engine)
    txn = pd.read_sql('SELECT * FROM fact_transaction', engine)
    engage = pd.read_sql('SELECT * FROM fact_engagement_events', engine)
    geo = pd.read_sql('SELECT * FROM dim_geography', engine)
    ind = pd.read_sql('SELECT * FROM dim_industry', engine) # note the typo 'indusrty'
    prod = pd.read_sql('SELECT * FROM dim_product', engine)

    print(f"Retrieved: cust {cust.shape}, usage {usage.shape}")
    return cust, usage, txn, engage, geo, ind, prod

#... rest of aggregation + feature engineering code stays same...

In [42]:
def aggregate_usage(df):
    df['Snapshot_Month'] = pd.to_datetime(df['Snapshot_Month'])
    agg = df.groupby('Customer_ID').agg({
        'Daily_Active_Users': ['mean', 'std', 'last'],
        'Feature_Adoption_Pct': ['mean', 'last'],
        'License_Utilization_Pct': 'mean',
        'API_Calls_Monthly': ['sum', 'mean'],
        'Uptime_Pct': 'mean',
        'Errors_Logged': 'sum',
        'Sessions_Total': 'sum'
    })
    agg.columns = ['_'.join(col).strip() for col in agg.columns.values]
    agg['dau_trend'] = df.groupby('Customer_ID')['Daily_Active_Users'].apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) > 1 else 0
    )
    return agg.reset_index()


In [43]:
def aggregate_transactions(df):
    agg = df.groupby('Customer_ID').agg({
        'Net_Revenue_USD': ['sum', 'mean', 'count'],
        'Payment_Delay_Days': ['mean', 'max'],
        'Discount_Pct': 'mean',
        'Add_On_Revenue_USD': 'sum',
        'Total_Billed_USD': 'sum'
    })
    agg.columns = ['_'.join(col) for col in agg.columns]
    agg['payment_reliability_score'] = 100 - agg['Payment_Delay_Days_mean'].clip(0, 100)
    return agg.reset_index()


In [44]:
def aggregate_engagement(df):
    agg = df.groupby('Customer_ID').agg({
        'Score': 'mean',
        'Sentiment': lambda x: (x == 'Positive').mean(),
        'SLA_Breached': 'sum',
        'Escalated_To_Manager': 'sum',
        'Resolution_Days': 'mean',
        'Event_Type': 'count'
    }).rename(columns={'Event_Type': 'total_events', 'Sentiment': 'positive_sentiment_ratio'})
    return agg.reset_index()


In [51]:
def feature_engineering(df):
    df['avg_monthly_revenue'] = df['Net_Revenue_USD_sum'] / df['Tenure_Months'].replace(0, 1)
    df['revenue_per_employee'] = df['ACV_USD'] / df['Employees'].replace(0, 1)
    df['licenses_unused'] = df['Licenses_Purchased'] - df['Daily_Active_Users_last']
    return df


In [52]:
def main():
    cust, usage, txn, engage, geo, ind, prod = load_from_postgres()
    
    # Note: Postgres usually converts column names to lowercase. Adjust if needed.
    # If your columns are CamelCase in DB, remove the.lower() calls below
    
    usage_agg = aggregate_usage(usage)
    txn_agg = aggregate_transactions(txn)
    engage_agg = aggregate_engagement(engage)
    
    master = cust.merge(usage_agg, on='Customer_ID', how='left')\
               .merge(txn_agg, on='Customer_ID', how='left')\
               .merge(engage_agg, on='Customer_ID', how='left')\
               .merge(geo, on='Geo_ID', how='left')\
               .merge(ind, on='Industry_ID', how='left')\
               .merge(prod, on='Product_ID', how='left')
    
    master = feature_engineering(master)
    master.to_csv(OUTPUT_DIR / 'master_dataset1.csv', index=False)
    
    # Also save back to Postgres for reuse
    master.to_sql('master_dataset', engine, if_exists='replace', index=False)
    print(f"Master dataset created: {master.shape}")
    print(f"Saved to CSV and Postgres table 'master_dataset'")


In [53]:
if __name__ == "__main__":
    main()

Retrieving 7 tables from PostgreSQL...
Retrieved: cust (5150, 51), usage (131268, 22)
Master dataset created: (5272, 108)
Saved to CSV and Postgres table 'master_dataset'
